# **🇹🇭 Constituency OCR: Free Spatial Matching with EasyOCR**

This notebook uses a completely **FREE, Offline, and Unlimited** Computer Vision approach. Instead of using massive Vision Language Models or paid APIs, we use **EasyOCR** (which natively supports Thai text) combined with a custom **Spatial Bounding Box Algorithm**.

### 🧠 How it works:
1. **EasyOCR** detects all text and numbers on the page along with their exact XY coordinates.
2. **Fuzzy Matching**: We look for the Party Name among the detected text boxes.
3. **Spatial Alignment**: Once we find the Party Name box, we scan horizontally to the right (on the same Y-axis row) to find the nearest box containing numbers. That number is our vote count!

This approach uses very little VRAM (~2GB) and never runs out of memory!

In [ ]:
!pip install -q easyocr rapidfuzz textdistance pandas pillow opencv-python

In [ ]:
import os
import re
import json
import glob
import textdistance
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from rapidfuzz import process, fuzz
import easyocr

# Initialize EasyOCR for Thai and English
print("Loading EasyOCR Reader into GPU...")
reader = easyocr.Reader(['th', 'en'], gpu=True)

## 1. Helper Functions

In [ ]:
DATA_DIR = './data'
IMG_DIR = os.path.join(DATA_DIR, 'images')
TEMPLATE_PATH = os.path.join(DATA_DIR, 'submission_template.csv')

sub_df = pd.read_csv(TEMPLATE_PATH)
doc_ids = sub_df['doc_id'].unique()

def get_images_for_doc(doc_id):
    """Reads all PNG pages for a given doc_id."""
    images = []
    base_img = os.path.join(IMG_DIR, f"{doc_id}.png")
    if os.path.exists(base_img):
        images.append(base_img)
    
    page = 2
    while True:
        page_img = os.path.join(IMG_DIR, f"{doc_id}_page{page}.png")
        if os.path.exists(page_img):
            images.append(page_img)
            page += 1
        else:
            break
    return images # Return file paths for EasyOCR

def clean_vote_string(s):
    """Removes non-digits and converts Thai numerals to Arabic."""
    thai_to_arabic = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
    s = str(s).translate(thai_to_arabic)
    s = re.sub(r'\D', '', s)
    return s if s else "0"

## 2. Spatial Core Extraction Algorithm

In [ ]:
def extract_votes_easyocr(doc_id, parties_list):
    image_paths = get_images_for_doc(doc_id)
    if not image_paths:
        return {p: "0" for p in parties_list}
    
    final_extracted_map = {p: "0" for p in parties_list}
    
    for img_path in image_paths:
        # Read all text from the page
        # detail=1 returns [(bounding_box, text, prob), ...]
        # bounding_box = [[x_tl, y_tl], [x_tr, y_tr], [x_br, y_br], [x_bl, y_bl]]
        try:
            results = reader.readtext(img_path, detail=1)
        except Exception as e:
            print(f"EasyOCR failed on {os.path.basename(img_path)}: {e}")
            continue
            
        # Process boxes into friendly formats
        boxes = []
        for bbox, text, prob in results:
            y_min = min([pt[1] for pt in bbox])
            y_max = max([pt[1] for pt in bbox])
            x_min = min([pt[0] for pt in bbox])
            x_max = max([pt[0] for pt in bbox])
            y_center = (y_min + y_max) / 2.0
            boxes.append({
                'text': text.strip(), 
                'clean_digits': clean_vote_string(text),
                'y_center': y_center, 
                'y_min': y_min, 
                'y_max': y_max, 
                'x_min': x_min,
                'x_max': x_max
            })
            
        # Now, for each party, try to find it on this page
        for party in parties_list:
            if final_extracted_map[party] != "0":
                continue # Already found on a previous page
                
            # Find best matching text box for the party name
            best_match = None
            best_score = 0
            for box in boxes:
                score = fuzz.partial_ratio(party, box['text'])
                if score > best_score:
                    best_score = score
                    best_match = box
                    
            # If we are reasonably confident we found the row (score > 80)
            if best_match and best_score > 80:
                # Find digits that are on the SAME ROW (Y overlap) and to the RIGHT (X > party_X)
                candidate_votes = []
                target_y_center = best_match['y_center']
                tolerance = (best_match['y_max'] - best_match['y_min']) * 1.5 # Allow slight tilt
                
                for box in boxes:
                    if box['clean_digits'] != "0" and len(box['clean_digits']) > 0:
                        # Is it on the same approximate Y axis?
                        if abs(box['y_center'] - target_y_center) < tolerance:
                            # Is it to the right?
                            if box['x_min'] > best_match['x_max']:
                                candidate_votes.append(box)
                
                if candidate_votes:
                    # Sort by X coordinate (closest to the right usually, but votes might be in a specific column)
                    # If there are multiple numbers, taking the one furthest right (often the 'total' or 'vote count' column)
                    candidate_votes.sort(key=lambda b: b['x_min'])
                    final_extracted_map[party] = candidate_votes[-1]['clean_digits']
                    
    return final_extracted_map

## 3. Evaluation on Sample Labels

In [ ]:
def evaluate_on_samples():
    label_files = glob.glob(os.path.join(DATA_DIR, "sample_labels", "*.json"))
    total_dist = 0
    total_rows = 0
    
    print(f"Evaluating Spatial OCR on {len(label_files)} ground truths...")
    
    for lpath in label_files:
        with open(lpath, 'r', encoding='utf-8') as f:
            gt_data = json.load(f)
            
        doc_id = os.path.basename(lpath).replace('.json', '')
        gt_map = {item['party']: str(item['votes']) for item in gt_data['results']}
        
        pred_map = extract_votes_easyocr(doc_id, list(gt_map.keys()))
        
        for party, gt_votes in gt_map.items():
            pred_votes = pred_map.get(party, "0")
            dist = textdistance.levenshtein(gt_votes, pred_votes)
            total_dist += dist
            total_rows += 1
            
    if total_rows > 0:
        print(f"EasyOCR Mean Levenshtein Distance: {total_dist / total_rows:.4f}")
        
# evaluate_on_samples()

## 4. Final Processing & Submission

In [ ]:
results_list = []

for doc_id in tqdm(doc_ids[:5], desc="Processing Documents (EasyOCR)"): # Remove [:5] for full run
    parties_to_find = sub_df[sub_df['doc_id'] == doc_id]['party_name'].tolist()
    
    extracted_map = extract_votes_easyocr(doc_id, parties_to_find)
    
    for party in parties_to_find:
        votes = extracted_map.get(party, "0")
                 
        results_list.append({
            'doc_id': doc_id,
            'party_name': party,
            'votes': votes
        })
        
for res in results_list:
    mask = (sub_df['doc_id'] == res['doc_id']) & (sub_df['party_name'] == res['party_name'])
    sub_df.loc[mask, 'votes'] = res['votes']

sub_df.to_csv('submission_easyocr.csv', index=False)
print("Submission successfully saved as submission_easyocr.csv!")
sub_df.head(10)